In [1]:
import sksurv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
import pandas as pd
import numpy as np
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import os
import kagglehub
from dataclasses import asdict
from sksurv.util import Surv
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from Utils.Config import Config, TrialResult
from Utils.ensemble_utils import collect_top_trial_oofs_from_configs
from Utils.utils import (
    set_seed,
    make_surv_y,
    create_features,
    HORIZONS
)
from Utils.ensemble_utils import find_ensemble_model
from Optuna_Experiment import SEEDS, MODEL_TYPES


Python:  3.11.14
OS:  Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
Scikit-learn:  1.8.0
Scikit-survival:  0.27.0
Metadata path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv
Train path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/train.csv
Test path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/test.csv



In [2]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')

In [3]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_processed = create_features(train_df)
test_processed = create_features(test_df)

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

event_horizon = np.array(HORIZONS).copy()
event_horizon[-1] = min(event_horizon[-1], train_processed['time_to_hit_hours'].max() - 1e-6)

print("Event horizons:", event_horizon)

Event horizons: [12.         24.         48.         66.99447313]


# Model Ensemble (differnt Config)

In [4]:
SEEDS = [42, 777, 1024, 2023]

print("SEEDS: ", SEEDS)
print("MODEL TYPES: ", MODEL_TYPES)

SEEDS:  [42, 777, 1024, 2023]
MODEL TYPES:  {'gbsa', 'coxnet'}


In [5]:
full_oof_result = collect_top_trial_oofs_from_configs(
    SEEDS, MODEL_TYPES, train_processed,
    HORIZONS, top_ratios={'gbsa': 0.01, 'coxnet': 0.1}
)
print("Final Dict length: ", len(full_oof_result))

Saving OOF predictions for top 3 trials to TOP_OOF/42/gbsa...
Best trial value: 0.9629674137093398
Best Tral Number: 106
[collect] seed=42, model=gbsa, added=3, total=3
Saving OOF predictions for top 30 trials to TOP_OOF/42/coxnet...
Best trial value: 0.9642943769968599
Best Tral Number: 247
[collect] seed=42, model=coxnet, added=30, total=33
Saving OOF predictions for top 3 trials to TOP_OOF/777/gbsa...
Best trial value: 0.9638562688053003
Best Tral Number: 185
[collect] seed=777, model=gbsa, added=3, total=36
Saving OOF predictions for top 30 trials to TOP_OOF/777/coxnet...
Best trial value: 0.9631892912495612
Best Tral Number: 197
[collect] seed=777, model=coxnet, added=30, total=66
Saving OOF predictions for top 3 trials to TOP_OOF/1024/gbsa...
Best trial value: 0.9611864713374448
Best Tral Number: 68
[collect] seed=1024, model=gbsa, added=3, total=69
Saving OOF predictions for top 30 trials to TOP_OOF/1024/coxnet...
Best trial value: 0.962907602370486
Best Tral Number: 168
[collec

# Select Ensemble Model

- 대회에 쓰이는 metric을 직접 최적화 하는 방식으로 진행

## 제약조건

1. 전체 trial중 30% (90개)
2. diversity를 어느정도 갖춘 core model [106,152,236,243]으로 시작
3. 이미 선택된 모델들과 corr이 0.995 이상으로 강하다면 skip
4. improve가 너무 적게 상승한다면 noise로 보고 skip
5. max model은 10개 정도 진행

## Config

In [6]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')
seed = 42
set_seed(seed)

train_df = pd.read_csv(train_path)
train_processed = create_features(train_df)

print("Metadata path: ", metadata_path)
print("Train path: ", train_path)
print("Test path: ", test_path)
print()

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

Metadata path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv
Train path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/train.csv
Test path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/test.csv



In [7]:
# init_model_list = [106,152,236,243]
best_key, best_item = max(
    full_oof_result.items(),
    key=lambda x: x[1]["value"]
)

init_model_list = [best_key]
max_pair_corr = 0.999
max_ensemble_corr = 0.998
min_improvement_score = 0.0001
max_model_num = 10
allow_duplicate = True
use_weight_grid_search = True

In [8]:
# 다른 모델도 후보로 넣으려면 
# .update 이용하여 딕셔너리 추가

# TODO 앙상블 모델당 weight도 추가로 grid search할수 있게 변경

ensemble_result = find_ensemble_model(
    oof_result=full_oof_result,
    label=y_full,
    max_ensemble_corr=max_ensemble_corr,
    max_pair_corr=max_pair_corr,
    min_imporvement_score=min_improvement_score,
    max_model_num=max_model_num,
    init_model_list=init_model_list,
    horizons=HORIZONS,
    verbose=True,
    allow_duplicate=allow_duplicate,
    use_weight_grid_search=use_weight_grid_search
)

===== Initial Ensemble =====
Initial models: [(42, 'coxnet', 247)]
Initial hybrid score: 0.964964
Allow duplicate: False
Max select per model: 3
Ensemble Candidate Model: 131

Current ensemble size: 1
Current hybrid score: 0.964964

[Trial (42, 'gbsa', 106)] 
score=0.965566 
improve=+0.000602 
pair_corr=0.98827 
ens_corr=0.98827 
duplicate=False
best_prev_weight=0.700 
 best_candidate_weight=0.300

[Trial (42, 'gbsa', 152)] 
score=0.965060 
improve=+0.000095 
pair_corr=0.99360 
ens_corr=0.99360 
duplicate=False
best_prev_weight=0.950 
 best_candidate_weight=0.050

[Trial (42, 'gbsa', 236)] 
score=0.965159 
improve=+0.000195 
pair_corr=0.99137 
ens_corr=0.99137 
duplicate=False
best_prev_weight=0.700 
 best_candidate_weight=0.300
[Trial (42, 'coxnet', 119)] skip (pair corr=0.99998 > 0.999)
[Trial (42, 'coxnet', 121)] skip (pair corr=1.00000 > 0.999)
[Trial (42, 'coxnet', 222)] skip (pair corr=0.99999 > 0.999)
[Trial (42, 'coxnet', 118)] skip (pair corr=0.99993 > 0.999)
[Trial (42, 'coxn

In [9]:
print(ensemble_result)

EnsembleModel(model_weights={(42, 'coxnet', 247): 0.5, (42, 'gbsa', 106): 0.5}, ensemble_score=None)


In [11]:
from Utils.ensemble_utils import search_ensemble_weight
ensemble_result = search_ensemble_weight(
    oof_result=full_oof_result, 
    model_dict=ensemble_result, 
    label=y_full, 
    eval_horizons=event_horizon
)
print(ensemble_result)

EnsembleModel(model_weights={(42, 'coxnet', 247): np.float64(0.782), (42, 'gbsa', 106): np.float64(0.21799999999999997)}, ensemble_score=MetricOuput(c_index=np.float64(0.9431103014243127), mean_brier=0.024738036386563668, hybrid_score=np.float64(0.9656164649566992)))
